In [1]:
# ============================================================
# BERT Fine-Tuning for Domain-Specific Text Classification
# Google Colab Version
# ============================================================

# 1. Install required libraries
!pip install -q transformers datasets evaluate accelerate torch

# ============================================================
# 2. Import Libraries
# ============================================================

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

# ============================================================
# 3. Create Domain-Specific Dataset
# ============================================================

data = {
    "text": [
        "The transformer model achieved excellent accuracy.",
        "Large Language Models are revolutionizing AI.",
        "The football team won the championship.",
        "The cricket match was exciting.",
        "Neural networks are widely used in deep learning.",
        "The player scored a brilliant goal.",
        "Machine learning improves decision making.",
        "The tennis tournament starts tomorrow."
    ],

    "label": [
        1,  # Technology
        1,  # Technology
        0,  # Sports
        0,  # Sports
        1,  # Technology
        0,  # Sports
        1,  # Technology
        0   # Sports
    ]
}

# Convert dictionary into Hugging Face Dataset
dataset = Dataset.from_dict(data)

print("Dataset:")
print(dataset)

# ============================================================
# 4. Load BERT Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

# ============================================================
# 5. Tokenization
# ============================================================

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(tokenize)

# Set dataset format for PyTorch
dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

print("\nTokenization completed successfully.")

# ============================================================
# 6. Load Pretrained BERT Model
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# ============================================================
# 7. Training Configuration
# ============================================================

training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

# ============================================================
# 8. Create Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

# ============================================================
# 9. Fine-Tune BERT Model
# ============================================================

print("\nStarting model training...")

trainer.train()

print("\nTraining completed successfully!")

# ============================================================
# 10. Save Fine-Tuned Model
# ============================================================

trainer.save_model("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

print("\nModel saved successfully!")

# ============================================================
# 11. Load Fine-Tuned Model for Prediction
# ============================================================

classifier = pipeline(
    "text-classification",
    model="./fine_tuned_model",
    tokenizer="./fine_tuned_model"
)

# ============================================================
# 12. Make Prediction
# ============================================================

text = "Generative AI models improve intelligent automation."

result = classifier(text)

# ============================================================
# 13. Define Class Labels
# ============================================================

labels = {
    "LABEL_0": "Sports",
    "LABEL_1": "Technology"
}

# ============================================================
# 14. Display Prediction
# ============================================================

print("\nPrediction")
print("-------------------------")
print("Input :", text)
print("Predicted Class :", labels[result[0]["label"]])
print("Confidence Score :", round(result[0]["score"], 3))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 950.3 kB/s eta 0:00:00
Dataset:
Dataset({
    features: ['text', 'label'],
    num_rows: 8
})


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting model training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)